# 04 Retention Analysis

**Purpose**

Define retention, prepare cohort/user-level tables, and evaluate repeat behavior. This notebook should support product decisions about bringing users back, but it should not contain ML modeling work.


## Setup And Sample Load

This starts with the same convenience sample for workflow development. Final retention metrics need the full observation window because retention is highly sensitive to time coverage.


In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display

pd.set_option("display.max_columns", 100)
pd.set_option("display.float_format", "{:.2f}".format)

DATA_DIR = Path("../data/raw")
RAW_FILES = sorted(DATA_DIR.glob("2019-*.csv"))
NROWS_PER_FILE = 100_000

RAW_FILES


In [ ]:
frames = []

for file_path in RAW_FILES:
    month_df = pd.read_csv(file_path, nrows=NROWS_PER_FILE)
    month_df["source_file"] = file_path.name
    frames.append(month_df)

df_raw = pd.concat(frames, ignore_index=True)

df_raw.head()


In [ ]:
df = df_raw.copy()

# Convert fields to analytical types.
df["event_time"] = pd.to_datetime(df["event_time"], utc=True)

id_columns = ["product_id", "category_id", "user_id", "user_session"]
df[id_columns] = df[id_columns].astype("string")

df["event_type"] = df["event_type"].astype("category")
df["category_code"] = df["category_code"].astype("string")
df["brand"] = df["brand"].astype("string")

df.dtypes


## Retention Readiness

Retention analysis needs enough observation time and user-level repeat behavior. Start by checking whether users have multiple sessions or activity across multiple days.


In [ ]:
session_size = (
    df.groupby("user_session")
      .size()
)

session_size.describe()


In [ ]:
sessions_per_user = (
    df.groupby("user_id")["user_session"]
      .nunique()
)

sessions_per_user.describe()


## User-Level Retention Feature Preparation

These features are descriptive retention-analysis inputs. They can later feed modeling in `05_retention_modeling.ipynb`, but model training and evaluation should stay out of this notebook.


In [ ]:
user_features = (
    df.groupby("user_id")
      .agg(
          first_event_time=("event_time", "min"),
          last_event_time=("event_time", "max"),
          active_days=("event_time", lambda s: s.dt.date.nunique()),
          sessions=("user_session", "nunique"),
          events=("event_time", "size"),
          products_viewed=("product_id", "nunique"),
          categories_viewed=("category_id", "nunique"),
          cart_events=("event_type", lambda s: (s == "cart").sum()),
          purchase_events=("event_type", lambda s: (s == "purchase").sum()),
          revenue=("price", lambda s: s[df.loc[s.index, "event_type"] == "purchase"].sum()),
      )
      .reset_index()
)

user_features["days_observed"] = (user_features["last_event_time"] - user_features["first_event_time"]).dt.days + 1
user_features["is_buyer"] = user_features["purchase_events"] > 0
user_features["is_repeat_session_user"] = user_features["sessions"] >= 2
user_features["is_multi_day_user"] = user_features["active_days"] >= 2

user_features.head()


In [ ]:
retention_readiness = pd.Series({
    "users": len(user_features),
    "repeat_session_users": user_features["is_repeat_session_user"].sum(),
    "repeat_session_user_pct": user_features["is_repeat_session_user"].mean() * 100,
    "multi_day_users": user_features["is_multi_day_user"].sum(),
    "multi_day_user_pct": user_features["is_multi_day_user"].mean() * 100,
    "buyers": user_features["is_buyer"].sum(),
    "buyer_pct": user_features["is_buyer"].mean() * 100,
}).round(2)

retention_readiness.to_frame("value")


In [ ]:
user_features[["sessions", "events", "products_viewed", "categories_viewed", "cart_events", "purchase_events", "revenue"]].describe().round(2)


## Cohort Preparation Placeholder

Add cohort logic after choosing the retention definition:

- Repeat session after first visit
- Multi-day return after first visit
- Repeat purchase after first purchase
- N-day or weekly cohort retention

Do not train predictive models here. Modeling belongs in `05_retention_modeling.ipynb`.
